In [6]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv(override=True)

username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")
host = "localhost"
port = 3306
database = "weather_db"

# Load and clean dataset
df = pd.read_csv("./datasets/weatherHistory.csv")

# Clean datetime
df['Formatted Date'] = pd.to_datetime(df['Formatted Date'], utc=True)
df['Date'] = df['Formatted Date'].dt.date
df['Hour'] = df['Formatted Date'].dt.hour
df['Month'] = df['Formatted Date'].dt.strftime('%Y-%m')  # avoids timezone warning

# Map Precip Type to integers
precip_mapping = {'rain': 1, 'snow': 2, 'sleet': 3, 'hail': 4, 'none': 0}
df['Precip Type'] = df['Precip Type'].map(lambda x: precip_mapping.get(str(x).lower(), 0))

# Calculate temperature difference
df['Temp_Diff'] = df['Temperature (C)'] - df['Apparent Temperature (C)']

# Categorize wind speed
def wind_category(speed):
    if speed < 5:
        return 0
    elif speed < 20:
        return 1
    else:
        return 2

df['Wind_Category'] = df['Wind Speed (km/h)'].apply(wind_category)

# Show unique values
precip_labels = {0:'None', 1:'Rain', 2:'Snow', 3:'Sleet', 4:'Hail'}
wind_labels = {0:'Calm', 1:'Moderate', 2:'High'}

print("Unique Precip Types:", [precip_labels[i] for i in df['Precip Type'].unique()])
print("Unique Wind Categories:", [wind_labels[i] for i in df['Wind_Category'].unique()])

# Show first few rows
print(df.head())

# Connect to MySQL
conn = mysql.connector.connect(
    host=host,
    user=username,
    password=password,
    database=database,
    port=port
)
cursor = conn.cursor()

# Create lookup tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS precip_type (
    id INT PRIMARY KEY,
    text VARCHAR(20)
)
""")
precip_values = [(0, 'None'), (1, 'Rain'), (2, 'Snow'), (3, 'Sleet'), (4, 'Hail')]
cursor.executemany("INSERT IGNORE INTO precip_type VALUES (%s, %s)", precip_values)

cursor.execute("""
CREATE TABLE IF NOT EXISTS wind_category (
    id INT PRIMARY KEY,
    text VARCHAR(20)
)
""")
wind_values = [(0, 'Calm'), (1, 'Moderate'), (2, 'High')]
cursor.executemany("INSERT IGNORE INTO wind_category VALUES (%s, %s)", wind_values)

# Create main weather table
cursor.execute("""
CREATE TABLE IF NOT EXISTS weather_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    formatted_date DATETIME,
    summary VARCHAR(50),
    precip_type INT,
    temperature FLOAT,
    apparent_temperature FLOAT,
    humidity FLOAT,
    wind_speed FLOAT,
    wind_bearing FLOAT,
    visibility FLOAT,
    loud_cover FLOAT,
    pressure FLOAT,
    daily_summary TEXT,
    date DATE,
    hour INT,
    temp_diff FLOAT,
    wind_category INT,
    month VARCHAR(7),
    FOREIGN KEY (precip_type) REFERENCES precip_type(id),
    FOREIGN KEY (wind_category) REFERENCES wind_category(id)
)
""")

# Prepare values for insertion
insert_cols = [
    'Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 
    'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)',
    'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover', 
    'Pressure (millibars)', 'Daily Summary', 'Date', 'Hour',
    'Temp_Diff', 'Wind_Category', 'Month'
]
values = [tuple(x) for x in df[insert_cols].to_numpy()]

# Insert data into MySQL
sql_insert = """
INSERT INTO weather_data (
    formatted_date, summary, precip_type, temperature, apparent_temperature, 
    humidity, wind_speed, wind_bearing, visibility, loud_cover, pressure, 
    daily_summary, date, hour, temp_diff, wind_category, month
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
cursor.executemany(sql_insert, values)
conn.commit()

print(f"{cursor.rowcount} weather records inserted successfully.")

# Close connection
cursor.close()
conn.close()
print("Database connection closed.")


Unique Precip Types: ['Rain', 'Snow', 'None']
Unique Wind Categories: ['Moderate', 'Calm', 'High']
             Formatted Date        Summary  Precip Type  Temperature (C)  \
0 2006-03-31 22:00:00+00:00  Partly Cloudy            1         9.472222   
1 2006-03-31 23:00:00+00:00  Partly Cloudy            1         9.355556   
2 2006-04-01 00:00:00+00:00  Mostly Cloudy            1         9.377778   
3 2006-04-01 01:00:00+00:00  Partly Cloudy            1         8.288889   
4 2006-04-01 02:00:00+00:00  Mostly Cloudy            1         8.755556   

   Apparent Temperature (C)  Humidity  Wind Speed (km/h)  \
0                  7.388889      0.89            14.1197   
1                  7.227778      0.86            14.2646   
2                  9.377778      0.89             3.9284   
3                  5.944444      0.83            14.1036   
4                  6.977778      0.83            11.0446   

   Wind Bearing (degrees)  Visibility (km)  Loud Cover  Pressure (millibars)  \
0  